# Extract

In [17]:
import muon as mu
import scanpy as sc

mdata = mu.read_h5mu("Dataset/muon.h5mu")

print(mdata) 

# for k in mdata.mod.keys():
#     print(k, mdata.mod[k])

adata = mdata.mod['rna']

print(adata)
print(adata.X.shape)

MuData object with n_obs × n_vars = 180794 × 33951
  2 modalities
    rna:	180794 x 33694
      obs:	'sample', 'batch_ID', 'Sample', 'Run', 'Subject', 'Status', 'DemuxletDropletType', 'IncludedInStudy', 'CellType', 'Cluster'
    protein:	180794 x 257
      obs:	'sample', 'batch_ID', 'Sample', 'Run', 'Subject', 'Status', 'DemuxletDropletType', 'IncludedInStudy', 'CellType', 'Cluster'
AnnData object with n_obs × n_vars = 180794 × 33694
    obs: 'sample', 'batch_ID', 'Sample', 'Run', 'Subject', 'Status', 'DemuxletDropletType', 'IncludedInStudy', 'CellType', 'Cluster'
(180794, 33694)


/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/homebrew/Caskroom/miniconda/base/envs/.venv/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Preprocessing

In [18]:
!git lfs install
!git clone https://huggingface.co/ctheodoris/Geneformer

git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
fatal: destination path 'Geneformer' already exists and is not an empty directory.


In [19]:
!pip install "transformers==4.57.1"


In [20]:
import scanpy as sc
import pandas as pd
!pip install mygene
import mygene

# obtain n_counts:
adata.obs["n_counts"] = adata.X.sum(axis=1)

rna = adata.X

print(adata.var.head())
print(adata.var.columns)
print(adata.var_names[:10])


Empty DataFrame
Columns: []
Index: [RP11-34P13.3, FAM138A, OR4F5, RP11-34P13.7, RP11-34P13.8]
Index([], dtype='object')
Index(['RP11-34P13.3', 'FAM138A', 'OR4F5', 'RP11-34P13.7', 'RP11-34P13.8',
       'RP11-34P13.14', 'RP11-34P13.9', 'FO538757.3', 'FO538757.2',
       'AP006222.2'],
      dtype='string')


In [ ]:
# Gene symbol → Ensembl ID via mygene (equivalent to R biomaRt — same Ensembl data source)
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()
gene_symbols = adata.var_names.tolist()

print(f"Querying {len(gene_symbols)} gene symbols...")

# as_dataframe=False → plain list of dicts, easier to handle multi-hits
hits = mg.querymany(
    gene_symbols,
    scopes='symbol,alias',
    fields='ensembl.gene',
    species='human',
    returnall=False,
    verbose=False,
)

# Build symbol → Ensembl mapping (keep first hit per query)
symbol_to_ensembl = {}
for h in hits:
    q = h.get('query')
    if q in symbol_to_ensembl or h.get('notfound'):
        continue
    ensembl = h.get('ensembl', {})
    if isinstance(ensembl, list):
        eid = ensembl[0].get('gene') if ensembl else None
    elif isinstance(ensembl, dict):
        eid = ensembl.get('gene')
    else:
        eid = None
    if eid:
        symbol_to_ensembl[q] = eid

# Merge onto adata.var (unmapped → NaN)
adata.var['ensembl_id'] = adata.var_names.map(symbol_to_ensembl)

n_mapped = adata.var['ensembl_id'].notna().sum()
print(f"Mapped: {n_mapped} / {len(adata.var)} genes ({n_mapped/len(adata.var)*100:.1f}%)")

# Save mapping CSV
mapping_df = adata.var[['gene_symbol', 'ensembl_id']].copy()
mapping_df.to_csv('Dataset/gene_ensembl.csv', index=True, index_label='var_name')
print("Saved: Dataset/gene_ensembl.csv")

Querying 33694 gene symbols...


In [ ]:
import os
import shutil
from huggingface_hub import hf_hub_download

REPO_ID = "ctheodoris/Geneformer"
GF_DIR = "Geneformer/geneformer"
MODEL_DIR = "Geneformer/Geneformer-V2-104M"

pkg_files = [
    "geneformer/gene_median_dictionary_gc104M.pkl",
    "geneformer/token_dictionary_gc104M.pkl",
    "geneformer/ensembl_mapping_dict_gc104M.pkl",
    "geneformer/gene_name_id_dict_gc104M.pkl",
]

model_files = [
    "Geneformer-V2-104M/model.safetensors",
]

def needs_download(path):
    if not os.path.exists(path):
        return True
    with open(path, 'rb') as f:
        return f.read(7) == b'version'

for hf_path in pkg_files:
    local = os.path.join(GF_DIR, os.path.basename(hf_path))
    if needs_download(local):
        print(f"Downloading {os.path.basename(hf_path)}...")
        src = hf_hub_download(repo_id=REPO_ID, filename=hf_path)
        shutil.copy(src, local)

for hf_path in model_files:
    local = os.path.join(MODEL_DIR, os.path.basename(hf_path))
    if needs_download(local):
        print(f"Downloading {os.path.basename(hf_path)} (large file)...")
        src = hf_hub_download(repo_id=REPO_ID, filename=hf_path)
        shutil.copy(src, local)

print("Ready 🔥")

## Embeddings

In [ ]:
import sys
import os
sys.path.insert(0, "Geneformer")

import scanpy as sc
from geneformer import TranscriptomeTokenizer

# Save h5ad with only mapped genes (ensembl_id required by Geneformer)
adata_gf = adata[:, adata.var['ensembl_id'].notna()].copy()
print(f"Genes with Ensembl ID: {adata_gf.n_vars}")

os.makedirs("input_data", exist_ok=True)
os.makedirs("tokenized", exist_ok=True)
adata_gf.write_h5ad("input_data/rna.h5ad")

# Tokenize
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={"CellType": "cell_type"},
    nproc=1,
    model_version="V2",
)
tk.tokenize_data(
    data_directory="input_data",
    output_directory="tokenized",
    output_prefix="gf_tokenized",
    file_format="h5ad",
)
print("Tokenization done → tokenized/gf_tokenized.dataset")

In [ ]:
from geneformer import EmbExtractor

emb_extractor = EmbExtractor(
    model_type="CellClassifier",
    num_classes=0,
    emb_mode="cell",
    cell_emb_style="mean_pool",
    filter_data=None,
    max_ncells=None,
    emb_layer=-1,
    forward_batch_size=100,
    nproc=4,
)

os.makedirs("embeddings", exist_ok=True)
embs = emb_extractor.extract_embs(
    model_directory="Geneformer/Geneformer-V2-104M",
    input_data_file="tokenized/gf_tokenized.dataset",
    output_directory="embeddings",
    output_prefix="gf_embs",
)

print(f"Embeddings shape: {embs.shape}")
print(embs.head())